# Урок 05 - Агентний RAG


## Налаштування

Цей зошит демонструє патерн Agentic RAG (Retrieval-Augmented Generation) за допомогою Microsoft Agent Framework.

**Передумови:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ваша кінцева точка служби Azure AI Search
- `AZURE_SEARCH_API_KEY` — ваш ключ API Azure AI Search
- Розгортання Azure OpenAI, налаштоване через змінні середовища
- Аутентифікація Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Що таке Agentic RAG?

Традиційний RAG дотримується фіксованого конвеєра: спочатку витягти документи, потім згенерувати відповідь. **Agentic RAG** йде далі, надаючи агенту автономію вирішувати **коли** і **як** отримувати інформацію.

З Agentic RAG агент може:
- **Приймати рішення**, чи потрібне витягнення інформації перед відповіддю на питання
- **Обирати**, до якого джерела даних або інструменту звернутися
- **Оцінювати** отримані результати і виконувати подальші витягнення, якщо перша спроба недостатня
- **Об’єднувати** інформацію з кількох етапів отримання у послідовну відповідь

Це робить агента більш гнучким і точним у порівнянні зі статичним конвеєром «витягнути — потім згенерувати».


## Створення інструменту пошуку

В Agentic RAG зовнішні джерела даних подані у вигляді **інструментів**, які агент може викликати за потреби. Це дає змогу агенту розглядати пошук як ще одну дію, яку він може виконати, а не як обов’язковий крок.

Нижче ми визначимо базу знань про подорожі та представимо її як інструмент, який агент може викликати для пошуку інформації про призначення.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Створення агента RAG

Тепер ми створюємо агента, якому наказано **завжди спершу отримувати інформацію перед відповіддю**. Агент використовує інструмент `search_travel_knowledge`, щоб базувати свої відповіді на базі знань, а не покладатися на власні тренувальні дані.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Ітеративний пошук — патерн «Виконавець-Перевіряючий»

Ключова перевага Agentic RAG — **ітеративний пошук**. Агент може виконувати кілька раундів пошуку, щоб перевірити, уточнити або розширити свої початкові результати — подібно до робочого процесу «виконавець-перевіряючий»:

1. **Крок виконавця**: агент отримує початкову інформацію та складає відповідь.
2. **Крок перевіряючого**: агент виконує додаткові пошуки, щоб перевірити деталі або заповнити прогалини.

Нижче агенту ставлять питання, яке вимагає порівняння кількох напрямків, що спонукає його виконати кілька пошукових запитів.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Резюме

У цьому уроці ви дізналися, як побудувати систему **Agentic RAG** за допомогою Microsoft Agent Framework:

- **Agentic RAG** дозволяє агентам автономно вирішувати, коли отримувати інформацію, роблячи пошук динамічним, а не фіксованим.
- **Інструменти як джерела даних**: Зовнішні бази знань (наприклад, Azure AI Search) упаковані як інструменти, які агент може викликати.
- **Ітеративний пошук**: Шаблон maker-checker дає змогу агенту виконувати кілька раундів пошуку — шукати, перевіряти і уточнювати — перед тим, як надати остаточну відповідь.

У виробництві ви заміните in-memory `TRAVEL_KNOWLEDGE_BASE` на реальний індекс Azure AI Search для обробки масштабного пошуку туристичних документів.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Застереження**:  
Цей документ було перекладено за допомогою сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, зверніть увагу, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критичної інформації рекомендується звертатися до професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
